### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
from groceries import Grocery

In [ ]:
grocery = Grocery.get_or_create("eggs")  # Fix 7: use .get(), not Grocery("eggs")    

In [ ]:
grocery.stock(10)

In [ ]:
grocery.consume(4)

In [ ]:
grocery.list_transactions()

In [ ]:
grocery.get_grocery_report()

### Now we write an MCP server and use it directly!

In [ ]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "smartfridge_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [ ]:
instructions = """You are a smart refrigerator that is able to inventory groceries, 
and answer questions about the inventory and the balance available. 

When told about current inventory, you MUST first call stock() for each item, 
then call consume() for any consumed quantities.

I have 10 eggs, 6 bananas and 10 breads. 
I have consumed 2 eggs, 6 bananas and 2 breads."""
request = "What's my balance of eggs, bananas, breads and milk?"
model = "gpt-4.1-mini"

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="grocery_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("grocery_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


### Now let's build our own MCP Client

In [ ]:
from groceries_client import get_groceries_tools_openai, read_groceries_resource, list_groceries_tools

mcp_tools = await list_groceries_tools()
print(mcp_tools)

In [ ]:
openai_tools = await get_groceries_tools_openai()
print(openai_tools)

In [ ]:
request = "What's my balance of eggs, bananas, breads and milk?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
context = await read_groceries_resource("eggs")
print(context)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>